# YOLOv11 Object Detection - Training & Inference

Skripsi: Sistem Deteksi Objek menggunakan YOLOv11

**Fitur:**
- Training model YOLOv11 custom
- Inference pada gambar
- Inference pada video
- Evaluasi model (mAP, Precision, Recall)

## 1. Installasi Dependencies

In [ ]:
!pip install ultralytics
!pip install roboflow

from ultralytics import YOLO
import os
import cv2
from google.colab import files
from IPython.display import display, Image, Video
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

print('Dependencies installed successfully!')

## 2. Upload Dataset

Upload `data.zip` yang berisi dataset dalam format YOLO.

In [ ]:
# Upload dataset
uploaded = files.upload()

# Unzip dataset
!unzip -q data.zip -d /content/dataset

# List files
!ls /content/dataset

## 3. Setup data.yaml

Buat file `data.yaml` untuk konfigurasi dataset.

In [ ]:
%%writefile /content/dataset/data.yaml
# Ganti path sesuai dengan struktur folder dataset Anda
path: /content/dataset
train: train/images
val: val/images

# Jumlah kelas
nc: 3

# Nama kelas (ganti sesuai dataset Anda)
names: ['class_0', 'class_1', 'class_2']

## 4. Training Model YOLOv11

In [ ]:
# Load model base YOLOv11s
model = YOLO('yolo11s.pt')

# Training
results = model.train(
    data='/content/dataset/data.yaml',
    epochs=400,
    imgsz=640,
    batch=16,
    patience=100,
    save=True,
    device=0  # GPU
)

## 5. Evaluasi Model

In [ ]:
# Load model terbaik
best_model = YOLO('/content/runs/detect/train/weights/best.pt')

# Evaluasi pada validation set
metrics = best_model.val(data='/content/dataset/data.yaml')

print(f'\n===== Hasil Evaluasi =====')
print(f'Precision    : {metrics.box.mp:.4f}')
print(f'Recall       : {metrics.box.mr:.4f}')
print(f'mAP@50       : {metrics.box.map50:.4f}')
print(f'mAP@50-95    : {metrics.box.map:.4f}')

In [ ]:
# Tampilkan grafik training
results_img = '/content/runs/detect/train/results.png'
if os.path.exists(results_img):
    img = mpimg.imread(results_img)
    plt.figure(figsize=(15, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Training Results')
    plt.show()

In [ ]:
# Tampilkan Confusion Matrix
cm_img = '/content/runs/detect/train/confusion_matrix_normalized.png'
if os.path.exists(cm_img):
    img = mpimg.imread(cm_img)
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Confusion Matrix (Normalized)')
    plt.show()

## 6. Inference pada Gambar

In [ ]:
# Upload gambar untuk inference
uploaded = files.upload()

# Jalankan inference
for filename in uploaded.keys():
    results = best_model(
        filename,
        conf=0.47,
        iou=0.45,
        imgsz=640
    )
    
    # Tampilkan hasil
    result = results[0]
    
    # Plot hasil
    annotated = result.plot()
    plt.figure(figsize=(12, 8))
    plt.imshow(annotated[..., ::-1])
    plt.axis('off')
    plt.title(f'Hasil Deteksi: {filename}')
    plt.show()
    
    # Tampilkan detail deteksi
    print(f'\n--- Deteksi pada {filename} ---')
    print(f'Jumlah objek: {len(result.boxes)}')
    for box in result.boxes:
        cls_name = result.names[int(box.cls[0])]
        conf = float(box.conf[0])
        print(f'  - {cls_name}: {conf*100:.1f}%')

## 7. Inference pada Video

In [ ]:
# Upload video
uploaded = files.upload()

for filename in uploaded.keys():
    # Inference pada video
    results = best_model(
        filename,
        conf=0.47,
        iou=0.45,
        imgsz=640,
        stream=True
    )
    
    # Simpan video hasil
    output_path = f'/content/{filename.split(".")[0]}_detected.mp4'
    
    # Proses video
    cap = cv2.VideoCapture(filename)
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (w, h))
    
    for result in results:
        annotated = result.plot()
        out.write(annotated)
    
    cap.release()
    out.release()
    
    print(f'Video tersimpan: {output_path}')

## 8. Export Model

In [ ]:
# Export model ke berbagai format
best_model.export(format='onnx')  # ONNX
# best_model.export(format='tflite')  # TFLite
# best_model.export(format='engine')  # TensorRT

print('Model exported successfully!')

In [ ]:
# Download model
files.download('/content/runs/detect/train/weights/best.pt')